# CatBoost L2 학습·검증 노트북

이 노트북은 저장된 `validation_predictions.csv`를 그리는 노트북이 아니라, 공유 데이터에서 **직접 CatBoost를 학습**하고 검증합니다.

- 통합 대상: 20개국 × 8개 HS6 제품
- 학습 타깃 기간: 2013Q1~2021Q4
- 검증 타깃 기간: 2022Q1~2025Q4, h=1~5
- 실험: 거시지표 포함·제외, 같은 검증 행 사용
- CatBoost 설정: `l2_leaf_reg=1.0`, seed=42
- 2026Q1~Q3 최종 테스트는 읽거나 사용하지 않습니다.

원천 통계의 과거 공개시점은 검증되지 않았으므로, 결과는 현재 수정치 기반의 1차 검증입니다.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import sys

import catboost
from catboost import CatBoostRegressor
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/shared/v001/manifest.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("프로젝트 폴더 또는 notebooks 폴더에서 실행하세요.")

DATA_DIR = ROOT / "data/shared/v001"
manifest_path = DATA_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
train = pd.read_parquet(DATA_DIR / "train.parquet")
validation = pd.read_parquet(DATA_DIR / "validation.parquet")

feature_columns = list(manifest["feature_columns"])
feature_contract = manifest["feature_contract"]
target = feature_contract["target"]
categorical_columns = list(feature_contract["categorical"])
forbidden = set(feature_contract["forbidden_feature_columns"])
assert not (set(feature_columns) & forbidden), "금지 입력 열이 feature_columns에 있습니다."
assert set(feature_columns).issubset(train.columns)
assert set(feature_columns).issubset(validation.columns)

model_config = yaml.safe_load((ROOT / "configs/models/catboost.yaml").read_text(encoding="utf-8"))
feature_config = yaml.safe_load((ROOT / "configs/features.yaml").read_text(encoding="utf-8"))
macro_base = set(feature_config["supervised"]["macro"]["columns"])
macro_columns = [c for c in feature_columns
                 if c in macro_base or c.removesuffix("_missing") in macro_base]
variants = {
    "with_macro": feature_columns,
    "without_macro": [c for c in feature_columns if c not in set(macro_columns)],
}

params = dict(model_config["params"])
params["l2_leaf_reg"] = 1.0
params.setdefault("allow_writing_files", False)
print("데이터 버전:", manifest["dataset_version"])
print("CatBoost:", catboost.__version__, "| l2_leaf_reg:", params["l2_leaf_reg"])
print("입력 열:", len(feature_columns), "| 거시 열 제외:", len(macro_columns))
print("train 후보 행:", len(train), "| validation 후보 행:", len(validation))

데이터 버전: v001
CatBoost: 1.2.10 | l2_leaf_reg: 1.0
입력 열: 57 | 거시 열 제외: 12
train 후보 행: 28800 | validation 후보 행: 12800


## 학습 시점과 입력 처리

각 검증 `origin_quarter`마다 모델을 새로 맞춥니다. 학습에는 해당 기준 분기까지 목표 정답이 확인된 train 행만 들어갑니다. 검증 정답이 없는 행은 점수 계산에서 제외합니다.

CatBoost 범주형 열은 manifest 기준의 국가·지역·HS 코드입니다. 범주 결측은 `__MISSING__` 문자열로 바꾸고, 수치 결측은 NaN으로 두어 CatBoost가 처리하게 합니다.

In [2]:
def quarter_index(values):
    parts = values.astype("string").str.extract(r"^(\d{4})Q([1-4])$")
    if parts.isna().any(axis=None):
        raise ValueError("분기 문자열이 YYYYQn 형식이 아닙니다.")
    return parts[0].astype(int) * 4 + parts[1].astype(int)


def make_features(frame, columns):
    x = frame.loc[:, columns].copy()
    cats = [c for c in categorical_columns if c in columns]
    for c in cats:
        x[c] = x[c].astype("string").fillna("__MISSING__").astype(str)
    for c in columns:
        if c not in cats:
            x[c] = pd.to_numeric(x[c], errors="coerce").astype("float64")
    return x, cats


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def score_group(frame):
    actual = frame[target].to_numpy(dtype=float)
    predicted = frame["prediction_usd"].to_numpy(dtype=float)
    error = predicted - actual
    denom = np.abs(actual).sum()
    smape_denom = np.abs(actual) + np.abs(predicted)
    smape = np.divide(200 * np.abs(error), smape_denom,
                       out=np.zeros_like(error), where=smape_denom != 0)
    return {
        "n_rows": len(frame),
        "actual_sum_usd": float(actual.sum()),
        "wape_pct": float(100 * np.abs(error).sum() / denom) if denom else None,
        "mae_usd": float(np.abs(error).mean()),
        "rmse_usd": float(np.sqrt(np.mean(error ** 2))),
        "smape_pct": float(smape.mean()),
    }


def make_metrics(predictions):
    records = []
    for variant, rows in predictions.groupby("variant", sort=True):
        group_specs = [
            ("overall", [], rows),
            ("operational_h2_h5", [], rows.loc[rows.horizon_quarters.between(2, 5)]),
            ("horizon", ["horizon_quarters"], rows),
            ("country", ["country_code"], rows),
            ("product", ["hs_code"], rows),
            ("country_horizon", ["country_code", "horizon_quarters"], rows),
            ("product_horizon", ["hs_code", "horizon_quarters"], rows),
        ]
        for dimension, keys, subset in group_specs:
            groups = [((), subset)] if not keys else subset.groupby(keys, dropna=False, sort=True)
            for key, group in groups:
                if not isinstance(key, tuple):
                    key = (key,)
                record = {"variant": variant, "dimension": dimension}
                record.update(dict(zip(keys, key)))
                record.update(score_group(group))
                records.append(record)
    return pd.DataFrame(records)

train = train.loc[train["split"].eq("train")].copy()
validation = validation.loc[validation["split"].eq("validation")].copy()
train["_target_q"] = quarter_index(train["target_quarter"])
train["_origin_q"] = quarter_index(train["origin_quarter"])
validation["_target_q"] = quarter_index(validation["target_quarter"])
validation["_origin_q"] = quarter_index(validation["origin_quarter"])
train_start = quarter_index(pd.Series([manifest["train_period"]["start"]])).iloc[0]
train_end = quarter_index(pd.Series([manifest["train_period"]["end"]])).iloc[0]
valid_start = quarter_index(pd.Series([manifest["validation_period"]["start"]])).iloc[0]
valid_end = quarter_index(pd.Series([manifest["validation_period"]["end"]])).iloc[0]
train = train.loc[train["_target_q"].between(train_start, train_end)]
validation = validation.loc[validation["_target_q"].between(valid_start, valid_end)
                            & validation["horizon_quarters"].between(1, 5)].copy()

has_target = validation[target].notna()
if "target_available" in validation:
    has_target &= validation["target_available"].fillna(False).astype(bool)
evaluation = validation.loc[has_target].copy()
print("정답 있는 검증 행:", len(evaluation), "| 정답 결측 제외:", len(validation) - len(evaluation))

정답 있는 검증 행: 11455 | 정답 결측 제외: 1345


## 모델 학습

아래 셀을 실행하면 origin 분기별로 `with_macro`와 `without_macro` 모델을 각각 학습하고 저장합니다. 한 origin에 들어가는 국가·제품은 하나의 pooled 모델에서 함께 학습합니다.

In [3]:
RUN_ID = datetime.now(timezone.utc).strftime("catboost_l2_nb_%Y%m%dT%H%M%SZ")
OUT = ROOT / "artifacts" / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

prediction_parts = []
model_records = []
origins = sorted(evaluation["_origin_q"].unique())
origin_labels = (evaluation[["_origin_q", "origin_quarter"]]
                 .drop_duplicates().set_index("_origin_q")["origin_quarter"].to_dict())

for origin_q in origins:
    origin = origin_labels[origin_q]
    fit_mask = (
        train[target].notna()
        & train["_target_q"].le(origin_q)
        & train["_origin_q"].le(origin_q)
    )
    fit_rows = train.loc[fit_mask]
    score_rows = evaluation.loc[evaluation["_origin_q"].eq(origin_q)]
    assert len(fit_rows) > 0
    assert fit_rows["_target_q"].le(origin_q).all()
    assert fit_rows["_origin_q"].le(origin_q).all()

    for variant, columns in variants.items():
        x_fit, cats = make_features(fit_rows, columns)
        x_score, _ = make_features(score_rows, columns)
        model = CatBoostRegressor(**params)
        model.fit(x_fit, fit_rows[target].astype(float), cat_features=cats)
        predicted = np.maximum(model.predict(x_score), 0.0)

        model_file = f"{variant}_{origin}.cbm"
        model.save_model(str(OUT / model_file))
        model_records.append({
            "variant": variant,
            "origin_quarter": origin,
            "model_file": model_file,
            "fit_rows": len(fit_rows),
            "fit_target_max_quarter": str(fit_rows["target_quarter"].max()),
            "feature_columns": columns,
            "categorical_columns": cats,
        })
        result = score_rows[["country_code", "hs_code", "origin_quarter",
                             "target_quarter", "horizon_quarters", target]].copy()
        result["variant"] = variant
        result["prediction_usd"] = predicted
        result["fit_origin_quarter"] = origin
        result["fit_rows"] = len(fit_rows)
        prediction_parts.append(result)
    print(f"{origin}: 학습 {len(fit_rows):,}행 / 검증 {len(score_rows):,}행", flush=True)

predictions = pd.concat(prediction_parts, ignore_index=True)
metrics = make_metrics(predictions)
predictions.to_csv(OUT / "validation_predictions.csv", index=False, encoding="utf-8-sig")
metrics.to_csv(OUT / "metrics.csv", index=False, encoding="utf-8-sig")

run_manifest = {
    "run_id": RUN_ID,
    "model_id": "catboost",
    "l2_leaf_reg": 1.0,
    "dataset_version": manifest["dataset_version"],
    "dataset_manifest_sha256": sha256(manifest_path),
    "train_period": manifest["train_period"],
    "validation_period": manifest["validation_period"],
    "validation_method": "expanding_origin; train target and row origin are <= fit origin",
    "target": target,
    "currency": "USD",
    "valuation": "FOB",
    "validation_candidate_rows": len(validation),
    "scored_rows_per_variant": int(evaluation.shape[0]),
    "missing_validation_targets_excluded": int(len(validation) - len(evaluation)),
    "feature_columns": feature_columns,
    "macro_columns_removed_for_without_macro": macro_columns,
    "categorical_columns": categorical_columns,
    "categorical_policy": "string values; null token __MISSING__; CatBoost native categories",
    "numeric_missing_policy": "NaN; CatBoost native numeric missing handling",
    "negative_prediction_policy": "clip at zero USD",
    "params": params,
    "versions": {"python": sys.version, "pandas": pd.__version__,
                 "numpy": np.__version__, "catboost": catboost.__version__},
    "models": model_records,
    "historical_asof_verified": bool(feature_contract.get("historical_asof_use_verified", False)),
}
(OUT / "run_manifest.json").write_text(
    json.dumps(run_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장 위치:", OUT)

2020Q4: 학습 20,565행 / 검증 137행
2021Q1: 학습 21,225행 / 검증 284행
2021Q2: 학습 21,910행 / 검증 425행
2021Q3: 학습 22,605행 / 검증 565행
2021Q4: 학습 23,290행 / 검증 703행
2022Q1: 학습 23,290행 / 검증 706행
2022Q2: 학습 23,290행 / 검증 704행
2022Q3: 학습 23,290행 / 검증 703행
2022Q4: 학습 23,290행 / 검증 705행
2023Q1: 학습 23,290행 / 검증 715행
2023Q2: 학습 23,290행 / 검증 718행
2023Q3: 학습 23,290행 / 검증 719행
2023Q4: 학습 23,290행 / 검증 723행
2024Q1: 학습 23,290행 / 검증 727행
2024Q2: 학습 23,290행 / 검증 728행
2024Q3: 학습 23,290행 / 검증 730행
2024Q4: 학습 23,290행 / 검증 584행
2025Q1: 학습 23,290행 / 검증 440행
2025Q2: 학습 23,290행 / 검증 294행
2025Q3: 학습 23,290행 / 검증 145행
저장 위치: d:\ogs\teamproject\five-stack\artifacts\catboost_l2_nb_20261104T081655Z


## 결과 요약

학습이 끝나면 아래 셀에서 전체 성능과 h=2~5 성능을 확인합니다. 행별 예측은 실행 결과 폴더의 `validation_predictions.csv`, 평가지표는 `metrics.csv`, 기준 분기별 모델은 `.cbm` 파일입니다.

In [4]:
summary = metrics.loc[metrics.dimension.isin(["overall", "operational_h2_h5", "horizon"]),
                       ["variant", "dimension", "horizon_quarters", "n_rows", "wape_pct",
                        "mae_usd", "rmse_usd", "smape_pct"]]
display(summary.sort_values(["variant", "dimension", "horizon_quarters"], na_position="first"))
print("모델 파일 수:", len(model_records))
print("검증 예측 행 수:", len(predictions))

,variant,dimension,horizon_quarters,n_rows,wape_pct,mae_usd,rmse_usd,smape_pct
2,with_macro,horizon,1.0,2291,43.470361,4.686300e+06,3.462030e+07,74.426346
3,with_macro,horizon,2.0,2291,45.277808,4.881151e+06,3.617075e+07,78.172674
4,with_macro,horizon,3.0,2291,46.954853,5.061944e+06,3.782152e+07,78.436484
5,with_macro,horizon,4.0,2291,49.401743,5.325729e+06,4.014175e+07,78.666445
6,with_macro,horizon,5.0,2291,49.249147,5.309279e+06,4.057396e+07,80.988379
1,with_macro,operational_h2_h5,NaN,9164,47.720888,5.144526e+06,3.871820e+07,79.065995
0,with_macro,overall,NaN,11455,46.870783,5.052880e+06,3.793405e+07,78.138066
177,without_macro,horizon,1.0,2291,45.673054,4.923760e+06,3.688961e+07,71.704044
178,without_macro,horizon,2.0,2291,47.187145,5.086986e+06,3.775524e+07,74.764271
179,without_macro,horizon,3.0,2291,48.906940,5.272387e+06,3.907758e+07,75.379502


모델 파일 수: 40
검증 예측 행 수: 22910
